# Verständnisdokument zu `min_variance.ipynb`

Dieses Notebook beschäftigt sich mit *min_variance.ipynb*, das NB wird hier step-by-step analysiert.

- Was passiert technisch in jeder Zelle?
- Welche Finance-Idee steckt dahinter?
- Welche Annahmen werden getroffen?
- Welche offenen Fragen ergeben sich für die Bachelorarbeit?

## 1. Kurzüberblick

**Ziel des Notebooks:**

*Den Prozess der Portfolio Optimisierung mit verschiedenen Möglichkeiten und Wegen aufzeigen*

**Aufbau:**

1. *Data Selection* -> Read the big source and gather a selected universe => Filter for sensful size, no nans
2. *Construct Universe* -> Further select by metric (narrow the universen by a specifc preference)
3. *Estiamte Covariance* -> Two Ways => **sample** & **Ledoit Wolf**
4. *Determine Portfolio Weights* -> different strategies => **naive**, **mcap**, **min_variance**
5. *Form and hold Portfolios* -> Simulate Investors for portfolios
6. *Evaluate Investment* -> compute & plot evaluaten   

**Erste Einordnung:**

| Bereich | Rolle im Notebook | Eigene Notizen |
|---|---|---|
| Daten einlesen | `read_feather()`; `US_data_panel`; | Rohe Daten gesamt |
| Daten filtern / vorbereiten | `filter_universe()`, `select_stock()` | basierend auf tag t_idx & Datenverfügbarkeit & metrik |
| Portfolio-Gewichte bestimmen | TODO | TODO |
| Rebalancing / Rolling Window | TODO | TODO |
| Performance messen | TODO | TODO |
| Visualisierung | TODO | TODO |

## 2. Cell-for-Cell-Analyse

Für jede Zelle (flexibel):

- **Was macht die Zelle technisch?**
- **Welche Finance- oder Statistik-Idee steckt dahinter?**
- **Welche Inputs werden benötigt?**
- **Welche Outputs entstehen?**
- **Was verstehe ich noch nicht?**

In [2]:
# Full DataFrame 

import pandas as pd

df = pd.read_feather("./Data/US_datastream/US_data_panel_filtered_0.15.feather")
df = df.sort_values(by="Date")

### Zelle 0: Überblick / Titel

**Titel:** `Overview - Portfolio Optimization Example`

**Was macht diese Zelle?**

Titel & Erklärung des Notebooks. Was wird gemacht (Datenselektion, Gewichtung, Portfoliobildung, Auswertung) 

**Welche Ideen?**

- Data Quality Management (Daten unvollständig o.ä.)
- Multiple Ansätze zur Gewichtung (Naive, macap, minVar) -> Vergleich interessant
- Lookahead & Lookback (basierend auf t, t+ oder t-)

### Zelle 1: Imports und Funktionsdefinitionen

**Titel:** `Methods, Packages and Class Definitions`

**Was macht die Zelle?**

=> Manages Imports and defines the helping structures we need for the actual process

*Bibliotheken:*

- numpy -> Berechnungen
- pandas -> Datenformat -> DataFrame
- sklearn -> Ledoit
- cvxpy -> Optmierung des Miminimerungs Problems
- tqdm -> Ladebalken
- functools -> Partielle Funktionen bzw. Params vorab belegen
- matplotlib -> Datenvisualisierung

*Definitionen:*

- `filter_universe()`

    - takes a panel (pd.DataFrame) and computes how much valid trading days and returns there are
    - if the fraction 1 - #valid_returns / #valid_trading_days is larger than a certain max_missing_frac than the stock is invalid
    - returns all valid sotcks as a List

    - => Filter out all stocks without proper data availability   

- `metric_sharpe()`

    - computes the pseudo Sharpe-Ratio for a given return_matrix

    - => used for stock selection

- `select_stocks()`

    - takes a return_matrix, a number_of_stocks and a scoring_metric
    - it then returns a list of the number_of_stocks stocks with the best metric_value

    - => actually selects the stocks 

- `panel_to_ret_matrix()`
    
    - just reshapes the given panel to a format where we have the stocks as colums and the returns by days as rows

    - => gives us the matrix format we need

- `covariance_estimator()`

    - takes a estimation method for the covariance matrix
    - computes the covariance matrix
    - makes sure that the diagonal works

    - => gives as the correct Covariance DataFrame

- `compute_weights()`

    - three methods to compute the portfolio weights (naive, mcap, min_variance)
    - naive is pretty simple
    - mcap gives higher weights to stocks with higher mcaps in the most recent value we have in the estimation window
    - min_variance actually makes the optimisation problem to get the weights accorsing to the risk in the stocks

    - => computes portfolio weights  

- `redristribute_weights()`

    - if a selected stock shows no data availabilty in the holding period we redistribute the weight it would have gotten to the other stocks in the portfolio

    - => makes sure we remain with our complete capital even when a stocks goes missing for whatever reason

- `Investor`
    
    - class that wraps our previous logic and is simulate the use of our logic
    - has a strategy (weights)
    - has a estimation method (covariance)
    - has get_returns() to compute portfolio_returns

        - selects the stocks in out portfolio out of the return matrix
        - multiples it with our weights 
        - with matrix multiplication '@' in python
        - => effectively here happens $r_p,t=\Sigma_i w_i * r_i,t$
    
    - => realises the actual portfolio investing part

- `evaluate_portfolio()`

    - takes returns and evaluates them
    - computes mean, std, psharpe and a couple more metrics annually (252 trading days in a year)

- `plot_portfolios()`

    - visualizes the whole thing


In [3]:
# Experimentierfeld zu Zelle 1
df.head(10)

,Date,Stock,AdjFactor,MTBV,MarketCAP,Close,High,Low,Open,ReturnIndex,UnadjClose,Volume,Return,DSCD,DelistingDate,prev_UnadjClose,BidAsk_Spread
17815282,1993-10-04,517890,1.000000,NaN,60.09299,11.000000,11.375000,11.000000,11.375000,1568.66200,11.000,7.20000,0.023256,517890,NaT,10.7500,0.027076
18853041,1993-10-04,542574,0.404085,1.359172,129.32520,9.799056,9.799056,9.597016,9.799056,427.91970,24.250,15.59077,0.000000,542574,NaT,23.7500,0.021076
10081737,1993-10-04,322810,0.333350,4.500755,864.81130,11.500570,11.917260,11.500570,11.750590,138.00000,34.500,170.09150,-0.021277,322810,2015-03-12,35.2500,0.015268
15774964,1993-10-04,510146,0.767273,NaN,194.76620,16.975910,17.359540,16.880000,17.263640,1041.17700,22.125,36.62318,-0.016667,510146,NaT,21.5000,0.001766
32514044,1993-10-04,905384,1.000000,1.788022,516.51590,35.750000,36.000000,34.750000,36.000000,570.59520,35.750,7.50000,0.028777,905384,NaT,34.7500,0.026069
32522008,1993-10-04,905387,0.444489,1.894211,692.25100,13.056850,13.112420,12.834610,12.890170,147.64570,29.375,352.98950,0.012931,905387,NaT,29.2500,0.007039
11177544,1993-10-04,326467,1.000000,NaN,51.49950,9.750000,10.000000,9.500000,9.750000,105.40540,9.750,75.70000,0.000000,326467,NaT,10.2500,0.081334
37222527,1993-10-04,923770,1.000000,0.537108,16.21100,14.500000,14.500000,14.375000,14.375000,515.70900,14.500,0.40000,0.017544,923770,NaT,14.0625,NaN
38759285,1993-10-04,938719,0.500000,2.413897,158.06440,6.562500,6.687500,6.437500,6.437500,65.21738,13.125,376.79960,0.000000,938719,NaT,12.6250,0.016575
32527931,1993-10-04,905403,1.000000,1.220014,2597.44100,19.750000,19.875000,19.500000,19.875000,495.61940,19.750,185.60000,-0.006289,905403,NaT,19.6250,0.012178


### Zelle 2: Datenimport, Parameter und Rebalancing-Schleife

**Titel:** `Determine weights and portdolio returns over time`

**Was macht die Zelle?**

simulates the actual use of the different strategies over the whole time period (starting two years after day 1), with rebalancing the portfolio of each strategy every month based on the last two years (lookback, backtest) 

**Step-by-Step:**

1. **Data**:

- read the main data set
- sort by dates -> unique & chronological -> loop over all *actual* days

- `estimation window` = amount of days for estimation (512 tradings days in 2 years)
- `holding period` = amount of days for evaluation (21 trading days in a month)

2. **Rabalancing Initialization** 

- `rebalance_indices` -> all date indices on which we rebuild our portfolio, recompute everything and start holding again (1 holding period apart) -> *Rolling Window*
- `Investors` -> capsulate the whole process in a more understandable way -> each investors runs on the same timeperiod and portfolio, makes them comparible

3. **Rebalancing loop**

- `for i, t_idx in enumerate(rebalance_indices)` -> i is a iteration index and t_idx is our rebalance index -> 0, 512; 1, 533; etc.
 
- Iteration:
    
    - we start with computing the estimation window:

        - `est_start` -> our current rebalance day - the size of our estimation window
        - `est_end` -> one day before the rebalance day -> rebalance day not in estimation window
        - `hold_end_idx` -> don't overflow incase we are at the end of all dates
        - `hold_dates` -> array of our holding period days
        - `panel` -> our estimation window cut out of the main DataFrame (*conditions in the DataFrame Access are super useful*)

    - Then, we filter out invalids:

        - we use our predefined logic `filter universe` with `max_missing_frac` to get valid_ids
        - reduce our DataFrame to only valids with nice tech -> `new_df = df[df[id].isin(valid_ids)]`
        - and transform the DF to our return_matrix format -> `panel_to_ret_matrix`
    
    - We then select stocks for our portfolio out of the valid ones:

        - with our predefined logic `select_stocks` with a `scoring_metric` and `n_stocks` -> *influence?*
        - reduce the valid panel to just selected ones with the same tech as before
        - also reduce our return matrix (*we did this earlier to select some stocks based on a metrix based on a return matrix*) -> with cool different (due to matrix form i guess) tech `new_df` = df.loc[:, selected_ids] 
        - we also sort the return matrix to match the order with our weights (I guess)
    
    - Now we loop over our individual strategies (Investors) -> same `ret_matrix`, `stock_universe`, `time period`

        - we estimate the covariance matrix with `covar_est()` based on a method (here we use either *numpy in-sample covariance computation* or *sklearn Ledoit Wold covariance estimation*) -> **so not implemented here -> I want to do that**
        - we then use the Covariance Matrix to get the portfolio weights with `investor.weights()` (partial function since different approaches)
        - we again check for NaNs and redestribute the weights if so
        - we then get the returns with `investor.get_returns()` (*Question*: Why do why reconstruct the panel and R out of the big DataFrame again? Can't we use the already computed ones?) 

In [4]:
# Experimentierfeld zu Zelle 2
import numpy as np

estimation_window, holding_period = 512, 21

actual_days = np.sort(df["Date"].unique())

T = len(actual_days)

rebalance_indices = range(estimation_window, T, holding_period)


In [5]:
# display Zelle

for i in rebalance_indices:
    pass # old

for i,j in enumerate(rebalance_indices):
    print(i, "|", j)

td = {}
td["BSP1"] = 1
td["BSP2"] = 2
td["BSP3"] = 3

for i,j in td.items():
    print(i, "|", j)


0 | 512
1 | 533
2 | 554
3 | 575
4 | 596
5 | 617
6 | 638
7 | 659
8 | 680
9 | 701
10 | 722
11 | 743
12 | 764
13 | 785
14 | 806
15 | 827
16 | 848
17 | 869
18 | 890
19 | 911
20 | 932
21 | 953
22 | 974
23 | 995
24 | 1016
25 | 1037
26 | 1058
27 | 1079
28 | 1100
29 | 1121
30 | 1142
31 | 1163
32 | 1184
33 | 1205
34 | 1226
35 | 1247
36 | 1268
37 | 1289
38 | 1310
39 | 1331
40 | 1352
41 | 1373
42 | 1394
43 | 1415
44 | 1436
45 | 1457
46 | 1478
47 | 1499
48 | 1520
49 | 1541
50 | 1562
51 | 1583
52 | 1604
53 | 1625
54 | 1646
55 | 1667
56 | 1688
57 | 1709
58 | 1730
59 | 1751
60 | 1772
61 | 1793
62 | 1814
63 | 1835
64 | 1856
65 | 1877
66 | 1898
67 | 1919
68 | 1940
69 | 1961
70 | 1982
71 | 2003
72 | 2024
73 | 2045
74 | 2066
75 | 2087
76 | 2108
77 | 2129
78 | 2150
79 | 2171
80 | 2192
81 | 2213
82 | 2234
83 | 2255
84 | 2276
85 | 2297
86 | 2318
87 | 2339
88 | 2360
89 | 2381
90 | 2402
91 | 2423
92 | 2444
93 | 2465
94 | 2486
95 | 2507
96 | 2528
97 | 2549
98 | 2570
99 | 2591
100 | 2612
101 | 2633
102 | 2654
1

### Zelle 3: Portfolio-Evaluation zusammenbauen und visualisieren

**Titel: Summary and Visualization**

**Was macht die Zelle?**

summarizes the holding process we simulated earlier 

-> `evaluate_portfolio`:

- takes a series of returns (daily) & n (total days, here a year)
- calaculates:
    - *mean*
    - *std*
    - *pseudo sharpe* = *sharpe without subtracting excess return*
    - *Value at risk*
    - *Expected Shortfall*
    - *Maximum Drawdown*

**Offene Fragen zu dieser Zelle:**

What is Value at Risk, Expected Shortfall and Maximum Drawdown? 

### Zelle 4: Visualisierung der Portfolio-Zeitreihen

**Inhalt:** `visualization = plot_portfolios(pd.concat(portfolio_timeseries, axis = 1))`

**Was macht die Zelle technisch?**

plots the evaluation (not with summary though, we plot the evaluation with daytime index)

-> `plot_portfolios`:

- TODO

**Warum ist eine Visualisierung zusätzlich zur Summary-Tabelle nützlich?**

TODO

**Offene Fragen zu dieser Zelle:**

TODO

### Zelle 5: Next possible steps

**Welche nächsten Schritte wären für die Bachelorarbeit sinnvoll?**

- => `transaction cost`
- => `self implementation for covariance estimation`
- => `more in detail stock selection`

**Mögliche Kategorien bis jetzt:**

- Robustere Kovarianzschätzung / Kovarianzschätzung allgemein -> Sensitivität von Schätzfehlern in der Kovarianzschätzung 
- Transaktionskosten -> in bestehendes Modell einbauen
- Turnover
- Alternative Strategien / Benchmarks
- Out-of-sample Evaluation -> einbauen
- Datenqualität und Survivorship Bias -> sehr interessant

## 3. Begriffsliste

| Begriff | Eigene Erklärung | Wo taucht es im Notebook auf? | Noch unklar? |
|---|---|---|---|
| Minimum Variance Portfolio | TODO | TODO | TODO |
| Kovarianzmatrix | TODO | TODO | TODO |
| Kovarianzschätzung | TODO | TODO | TODO |
| Shrinkage | TODO | TODO | TODO |
| Ledoit-Wolf | TODO | TODO | TODO |
| Rolling Window | TODO | TODO | TODO |
| Estimation Window | TODO | TODO | TODO |
| Holding Period | TODO | TODO | TODO |
| Rebalancing | TODO | TODO | TODO |
| Turnover | TODO | TODO | TODO |
| Transaktionskosten | TODO | TODO | TODO |
| Out-of-sample Evaluation | TODO | TODO | TODO |
| Benchmark | TODO | TODO | TODO |
| Sharpe Ratio | TODO | TODO | TODO |
| Volatilität | TODO | TODO | TODO |

## 4. Eigene Zusammenfassung

**In einem Satz:**

We iteratively go through time and look back to determine a `stock_universe`, this is then used to form a portfolio which then gets held for a `holding_period` and evaluated.  

**In einem Absatz:**

We iteratively go through a time period. At each day `t_idx` we look back 2 years and form a `stock_universe` based on a certain metric. We then proceed with forming portfolios of different strategies based on our lookback information. We simulated a `holding_period` before rebalancing the individual portfolios. After we ran through the whole simulation we evaluate the performance of each strategy and also visualize the results.


**Welche Fragen möchte ich als nächstes klären?**

1. What is *Value at Risk*, *Expected Shortfall* and *Maximum Drawdown*?
2. (in last "NaN-Check" in second loop), why do we even can encounter NaNs here?
3. in `ret_selected = ret_selected.loc[:, ret_selected.columns.sort_values()]` why do we need to sort here? Is it for the matrix calculations?
    - There are multiple sections where I can image why we sort, but I'd love to understand it in more detail (e.g. in rebalancing)
4. `Sigma = Sigma + ridge * np.eye(Sigma.shape[0])` I remember the explanation so I know why we do this, but honestly I don't understand it in detail
5. What's the Difference between Backtest & Lookback? I understand lookback pretty well, but is Backtest the same thing? Does it also mean we use the past data for estimation?

**Was muss unbedingt noch gemacht werden?**

> review the pandas DataFrame syntax/logic

> review/train linear algebra

> review and look at the underlying mathematics in more detail

> logic like `pf_returns = ret_mat.loc[:, w.index].values @ w.values` should be fluent

> get into the cvxpy/optimization logic

> I have to review python data visualization